# NVDA exploratory analysis

Why this notebook exists: the modelling results in `reports/EVALUATION.md` say
that seven models found no exploitable signal in daily returns, and that
volatility is the one quantity that measurably changed between the training and
test periods. Both of those are *conclusions from models*.

This notebook establishes the same two facts **from the data alone**, before any
model is fitted. If the raw series already tells you returns are unpredictable
and volatility is not, then the modelling result is confirmation rather than
surprise — and the recommendation to model volatility next is grounded in the
data rather than in a post-hoc reading of a leaderboard.

Run from the project root so `src` is importable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import PATHS
from src.split import chronological_split

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

raw = pd.read_csv(PATHS.raw, parse_dates=["Date"]).sort_values("Date")
print(f"{len(raw)} bars, {raw.Date.min().date()} to {raw.Date.max().date()}")
print(f"close ${raw.Close.min():.4f} -> ${raw.Close.max():.2f} "
      f"({raw.Close.max()/raw.Close.min():,.0f}x)")

## 1. The series is not stationary; its returns are

An augmented Dickey-Fuller test on the log price cannot reject a unit root. On
log returns it rejects overwhelmingly. This is the formal version of "model
returns, not levels" — every modelling choice downstream follows from it.

In [ ]:
from statsmodels.tsa.stattools import adfuller

log_ret = np.log(raw.Close).diff().dropna()
# result_object=False keeps the tuple return that statsmodels 0.16
# will change; pinning it here so the cell does not start warning.
p_level = adfuller(np.log(raw.Close.values), autolag="AIC",
                   result_object=False)[1]
p_ret = adfuller(log_ret.values, autolag="AIC", result_object=False)[1]

print(f"ADF p-value   log price : {p_level:.4f}   (cannot reject a unit root)")
print(f"ADF p-value   log return: {p_ret:.2e}  (stationary)")

## 2. The central result: returns have no memory, volatility does

Autocorrelation of returns against autocorrelation of *squared* returns. The
first is the predictability of direction; the second is the predictability of
magnitude.

This single comparison is the empirical foundation of the entire project.

In [ ]:
from statsmodels.tsa.stattools import acf

a_ret = acf(log_ret, nlags=20, fft=True)[1:]
a_sq = acf(log_ret**2, nlags=20, fft=True)[1:]

fig, ax = plt.subplots(figsize=(9, 4))
lags = np.arange(1, 21)
ax.bar(lags - 0.2, a_ret, width=0.4, label="returns")
ax.bar(lags + 0.2, a_sq, width=0.4, label="squared returns")
ci = 1.96 / np.sqrt(len(log_ret))
ax.axhline(ci, color="red", ls="--", lw=1, label="95% band")
ax.axhline(-ci, color="red", ls="--", lw=1)
ax.set_xlabel("lag (trading days)"); ax.set_ylabel("autocorrelation")
ax.set_title("Returns are white noise. Squared returns are not.")
ax.legend(frameon=False)
plt.show()

print("lag   return ACF   squared-return ACF")
for i in range(5):
    print(f"{i+1:>3}   {a_ret[i]:>10.4f}   {a_sq[i]:>17.4f}")

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

lb_r = acorr_ljungbox(log_ret, lags=[10], return_df=True)["lb_pvalue"].iloc[0]
lb_r2 = acorr_ljungbox(log_ret**2, lags=[10], return_df=True)["lb_pvalue"].iloc[0]

print(f"Ljung-Box(10) on returns         p = {lb_r:.4f}")
print(f"Ljung-Box(10) on squared returns p = {lb_r2:.2e}")

**p = 0.053 versus p = 3e-119.**

There is no detectable linear structure in the direction of daily returns. There
is overwhelming structure in their magnitude — volatility clustering, the
best-documented regularity in daily equity data.

Everything in `reports/EVALUATION.md` follows from this. Seven models were asked
to predict the quantity with no structure. The one with structure was never
modelled. That is the recommendation in §10, and it is visible here in two
autocorrelation functions.

## 3. Fat tails

Daily returns are far from normal, which is why the 80% prediction interval in
the API is computed from realised volatility rather than assumed Gaussian.

In [ ]:
sigma = log_ret.std()
beyond_5 = int((log_ret.abs() > 5 * sigma).sum())
expected = len(log_ret) * 2 * 2.87e-7

print(f"skew      {log_ret.skew():.3f}")
print(f"kurtosis  {log_ret.kurtosis():.2f}   (normal = 0)")
print(f"moves beyond 5 sigma: {beyond_5} observed, {expected:.3f} expected under normality")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(log_ret, bins=120, density=True, alpha=0.7, label="observed")
x = np.linspace(log_ret.min(), log_ret.max(), 400)
ax.plot(x, np.exp(-0.5*(x/sigma)**2)/(sigma*np.sqrt(2*np.pi)), "r--", label="normal")
ax.set_yscale("log"); ax.set_xlabel("daily log return")
ax.set_title("Log scale: the tails are orders of magnitude heavier than normal")
ax.legend(frameon=False)
plt.show()

## 4. The level gap that forces return-based targets

The chronological split puts training and test in different price regimes
entirely. A tree trained on levels cannot predict beyond its training range; a
min-max-scaled network maps every test price outside the interval it learnt.

In [ ]:
feats = pd.read_csv(PATHS.features, parse_dates=["Date"])
s = chronological_split(feats, save=False)

print(f"train  max ${s.train.Close.max():>7.2f}")
print(f"val    max ${s.val.Close.max():>7.2f}")
print(f"test   max ${s.test.Close.max():>7.2f}   "
      f"({s.test.Close.max()/s.train.Close.max():.0f}x the training maximum)")

fig, ax = plt.subplots(figsize=(10, 4))
for name, part, c in (("train", s.train, "#3a7ca5"), ("val", s.val, "#f2a541"),
                      ("test", s.test, "#76b900")):
    ax.plot(part.Date, part.Close, lw=0.9, color=c, label=f"{name} (n={len(part)})")
ax.set_yscale("log"); ax.set_ylabel("close, log scale")
ax.set_title("Three price regimes, one model")
ax.legend(frameon=False)
plt.show()

## 5. Volatility across the split

The drift report finds `volatility_21` as the worst-drifted feature (PSI 1.483,
shifted downward). Here is why: the training window contains the dot-com
collapse and 2008. The test window contains neither.

In [ ]:
vol = log_ret.rolling(21).std() * np.sqrt(252) * 100
vol.index = raw.Date.iloc[1:]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(vol.index, vol, lw=0.7, color="#3a7ca5")
ax.axvline(s.train.Date.max(), color="black", ls="--", lw=1)
ax.axvline(s.val.Date.max(), color="black", ls="--", lw=1)
ax.set_ylabel("annualised 21-day volatility (%)")
ax.set_title("Training spans 2000 and 2008; the test window does not")
plt.show()

for name, part in (("train", s.train), ("val", s.val), ("test", s.test)):
    w = vol.loc[part.Date.min():part.Date.max()]
    print(f"{name:>5}  mean {w.mean():5.1f}%   p95 {w.quantile(.95):6.1f}%   max {w.max():6.1f}%")

## What this notebook establishes

1. Log prices carry a unit root; log returns do not. Model returns.
2. Returns show no linear autocorrelation (Ljung-Box p = 0.05). Squared returns
   show overwhelming autocorrelation (p = 3e-119). **Direction is
   unpredictable; magnitude is not.**
3. Returns are heavily fat-tailed, so Gaussian prediction intervals understate
   risk.
4. The train/test level gap is 33x, which is why tree and network models must
   learn returns rather than prices.
5. Volatility is materially lower in the test window, which is what the drift
   report detects as PSI 1.483 on `volatility_21`.

Points 2 and 5 together are the argument for modelling volatility next — made
here from the data, independently of any model's leaderboard position.